In [1]:
from time import sleep

import pandas as pd

from IPython.display import display

from pulao.constant import SwingPointLevel, SwingPointType
from pulao.trend import TrendManager
from pulao.swing import SwingManager
from pulao.bar import SBar, SBarManager, CBarManager, CBar
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import BarData

import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pl.read_csv("../dataset/I8888.XDCE_60m.csv", try_parse_dates=True)
df = df.head(100)  # test

sbar_list = []
columns = df.columns

for idx, row in enumerate(df.iter_rows()):
    row_dict = dict(zip(columns, row))
    # datetime,open,close,high,low,volume,money,open_interest,signal
    datetime = row_dict["datetime"]
    open = row_dict["open"]
    close = row_dict["close"]
    high = row_dict["high"]
    low = row_dict["low"]
    volume = row_dict["volume"]
    money = row_dict["money"]
    open_interest = row_dict["open_interest"]

    bar = BarData(
        gateway_name="ctp-test",
        symbol="i8888",
        exchange=Exchange.SHFE,
        interval=Interval.MINUTE,
        datetime=datetime,
        open_price=open,
        close_price=close,
        high_price=high,
        low_price=low,
        volume=volume,
        open_interest=open_interest,
        turnover=money,
    )
    sbar = SBar(index= len(sbar_list), symbol=bar.symbol, exchange=bar.exchange.value, interval=bar.interval.value,datetime=datetime,turnover=money,open_price=open,close_price=close,high_price=high,low_price=low,volume=volume)

    sbar_list.append(sbar)
# 模拟行情数据接收
sbar_manager = SBarManager()
cbar_manager = CBarManager(sbar_manager=sbar_manager)
swing_manager = SwingManager(cbar_manager=cbar_manager)
trend_manager = TrendManager(swing_manager=swing_manager)

# 流入模拟数据
for sbar in sbar_list:
    sbar_manager.append(sbar)

#
# region 2. Plotly - Cbar
#
df_pandas2 = cbar_manager.df_cbar.to_pandas()
df_pandas2["datetime"] = pd.date_range("2025-01-01", periods=df_pandas2.shape[0], freq="h")
df_pandas2["open_price"] = df_pandas2["high_price"]
df_pandas2["close_price"] = df_pandas2["low_price"]

fig2 = make_subplots(
    rows=1, cols=1
)
fig2.add_trace(go.Candlestick(
    x=df_pandas2['datetime'],
    open=df_pandas2['open_price'],
    high=df_pandas2['high_price'],
    low=df_pandas2['low_price'],
    close=df_pandas2['close_price'],
    name='K线',
), row=1, col=1)

df_swing_point_low2 = df_pandas2[(df_pandas2['swing_point_type'] == SwingPointType.LOW) & (df_pandas2["swing_point_level"] == SwingPointLevel.MAJOR)]

fig2.add_trace(go.Scatter(
    x=df_swing_point_low2['datetime'],
    y=df_swing_point_low2['low_price'] * 0.995,   # 放在K线最低价下方一点，不遮挡蜡烛
    mode='markers',
    name='swing point low',
    marker=dict(
        symbol='triangle-up',
        size=14,
        color='#1E90FF',
    ),
    hovertemplate=
        "<b>波段低点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)


df_swing_point_high2 = df_pandas2[(df_pandas2['swing_point_type'] == SwingPointType.HIGH) & (df_pandas2["swing_point_level"] == SwingPointLevel.MAJOR)]

fig2.add_trace(go.Scatter(
    x=df_swing_point_high2['datetime'],
    y=df_swing_point_high2['high_price'] * 1.005,  # 放在K线最高价上方一点
    mode='markers',
    name='swing point high',
    marker=dict(
        symbol='triangle-down',
        size=14,
        color='#FF4500',
    ),
    hovertemplate=
        "<b>波段高点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)


df_swing_point_low21 = df_pandas2[(df_pandas2['swing_point_type'] == SwingPointType.LOW) & (df_pandas2["swing_point_level"] == SwingPointLevel.MINOR)]

fig2.add_trace(go.Scatter(
    x=df_swing_point_low21['datetime'],
    y=df_swing_point_low21['low_price'] * 0.995,   # 放在K线最低价下方一点，不遮挡蜡烛
    mode='markers',
    name='swing point low - 1',
    marker=dict(
        symbol='triangle-up',
        size=14,
        color='#333333',
    ),
    hovertemplate=
        "<b>波段低点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)


df_swing_point_high21 = df_pandas2[(df_pandas2['swing_point_type'] == SwingPointType.HIGH) & (df_pandas2["swing_point_level"] == SwingPointLevel.MINOR)]

fig2.add_trace(go.Scatter(
    x=df_swing_point_high21['datetime'],
    y=df_swing_point_high21['high_price'] * 1.005,  # 放在K线最高价上方一点
    mode='markers',
    name='swing point high - 1',
    marker=dict(
        symbol='triangle-down',
        size=14,
        color='#333333',
    ),
    hovertemplate=
        "<b>波段高点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)



# title = 'CBar Chart - K线合并处理'
title = 'CBar Chart - 分形重叠处理'
# title = 'CBar Chart - 次级别处理'
# title = 'CBar Chart - 连续同级别同向高低点处理'
fig2.update_layout(title=title,
    height=900,
    hovermode='x unified',    # X轴悬停联动虚线
)

fig2.update_xaxes(
    showgrid=False,
)

fig2.update_yaxes(
    showgrid=False,
)

fig2.show()


# endregion

ooooooooooooooooooooooooooooooooooooooooooooooooooooooooo
ooooooooooooooooooooooooooooooooooooooooooooooooooooooooo
ooooooooooooooooooooooooooooooooooooooooooooooooooooooooo
ooooooooooooooooooooooooooooooooooooooooooooooooooooooooo
ooooooooooooooooooooooooooooooooooooooooooooooooooooooooo
ooooooooooooooooooooooooooooooooooooooooooooooooooooooooo
ooooooooooooooooooooooooooooooooooooooooooooooooooooooooo
ooooooooooooooooooooooooooooooooooooooooooooooooooooooooo
ooooooooooooooooooooooooooooooooooooooooooooooooooooooooo
ooooooooooooooooooooooooooooooooooooooooooooooooooooooooo
ooooooooooooooooooooooooooooooooooooooooooooooooooooooooo
ooooooooooooooooooooooooooooooooooooooooooooooooooooooooo
ooooooooooooooooooooooooooooooooooooooooooooooooooooooooo
ooooooooooooooooooooooooooooooooooooooooooooooooooooooooo
ooooooooooooooooooooooooooooooooooooooooooooooooooooooooo
ooooooooooooooooooooooooooooooooooooooooooooooooooooooooo
ooooooooooooooooooooooooooooooooooooooooooooooooooooooooo
oooooooooooooo

Traceback (most recent call last):
  File "D:\PycharmProjects\pulao\pulao\events.py", line 25, in notify
    fn(event_type, payload)
    ~~^^^^^^^^^^^^^^^^^^^^^
  File "D:\PycharmProjects\pulao\pulao\swing\swing_manager.py", line 32, in _on_cbar_created
    self.detect()
    ~~~~~~~~~~~^^
  File "D:\PycharmProjects\pulao\pulao\swing\swing_manager.py", line 36, in detect
    self._adjust_swing_point_level()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "D:\PycharmProjects\pulao\pulao\swing\swing_manager.py", line 69, in _adjust_swing_point_level
    self._process_fractal_overlap()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "D:\PycharmProjects\pulao\pulao\swing\swing_manager.py", line 89, in _process_fractal_overlap
    df_fractals = self.df_cbar.tail(
                  ^^^^^^^^^^^^
AttributeError: 'SwingManager' object has no attribute 'df_cbar'
Traceback (most recent call last):
  File "D:\PycharmProjects\pulao\pulao\events.py", line 25, in notify
    fn(event_type, payload)
    ~~^^^^^^

In [2]:
cbar_manager.df_cbar.filter(pl.col("swing_point_level") != SwingPointLevel.NONE)

index,start_index,end_index,high_price,low_price,swing_point_type,swing_point_level,swing_point_level_origin
u32,u32,u32,f32,f32,i16,u8,u8
1,2,2,939.052002,930.297974,-1,2,2
3,4,5,955.56897,948.888,1,2,2
4,6,6,950.607971,946.338989,-1,2,2
6,9,10,961.794983,952.804016,1,2,2
8,12,12,953.807007,947.198975,-1,2,2
…,…,…,…,…,…,…,…
63,88,89,943.90802,936.588013,1,2,2
65,91,91,937.406006,933.703003,-1,2,2
67,93,93,955.924988,937.934021,1,2,2
